# 🧪 معمل التجارب: قاعدة بيانات المتجهات السحابية DataStax Astra DB (Lecture 35)
أهلاً بك في معملك التطبيقي! 🚀

**فكرة المعمل ببساطة:**
في هذا الدرس، ننتقل من مخازن المتجهات المحلية (Local Stores) إلى أول **قاعدة بيانات متجهات سحابية متكاملة (Cloud Vector Database)**:
منصة **DataStax Astra DB** المبنية على محرك **Apache Cassandra** الأسطوري، والمعروف بقدرته على التوسع الهائل وزمن استجابة يقترب من الصفر (Near Zero Latency).

في هذا المعمل ستتعلم عملياً:
1. **ربط التطبيق بالسحابة:** عبر مفتاحي `ASTRA_DB_API_ENDPOINT` و `ASTRA_DB_APPLICATION_TOKEN`.
2. **إنشاء الفهرس والمجموعة (`AstraDBVectorStore`):** وإدارة المتجهات والمستندات على السحابة مباشرة.
3. **البحث المتقدم والتصفية:** البحث الدلالي العادي، التصفية بالـ Metadata، واستخراج درجات الثقة (Scores).
4. **المسترجعات المتقدمة:** مسترجع عتبة الثقة (`similarity_score_threshold`) ومسترجع تنوع النتائج (`mmr`).
5. **بناء مسار RAG سحابي متكامل:** باستخدام **LCEL** ونموذج **Groq** فائق السرعة.

> 💡 **طريقة التدريب في المعمل:**
> اقرأ الشرح المبسط ⬅️ شوف الكود المرجعي الجاهز ⬅️ واكتب الكود وجربه بإيدك في الخلية الفارغة المخصصة لك تحت كل خطوة!

### 1. استيراد المكتبات وتحميل إعدادات الاتصال السحابي
**الفكرة ببساطة:**
نحتاج إلى معلومتين أساسيتين من لوحة تحكم Astra DB (`astra.datastax.com`):
1. **`ASTRA_DB_API_ENDPOINT`**: رابط الـ API الخاص بقاعدتك (ينتهي بـ `.apps.astra.datastax.com`).
2. **`ASTRA_DB_APPLICATION_TOKEN`**: التوكن السري (يبدأ بـ `AstraCS:...`).
نقوم بتحميلهما من ملف `.env`، مع توفير وضع محاكاة ذكي في حال لم تكن قد أنشأت حساباً بعد لضمان تشغيل وتجربة الكود فوراً.

In [ ]:
# 1. import standard libraries and load environment variables
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# load astra credentials from .env
astra_db_api_endpoint = os.getenv("ASTRA_DB_API_ENDPOINT")
astra_db_application_token = os.getenv("ASTRA_DB_APPLICATION_TOKEN")

# check credential presence
has_credentials = bool(
    astra_db_api_endpoint 
    and astra_db_application_token 
    and not astra_db_api_endpoint.startswith("https://your-")
)

print("✅ Libraries imported successfully!")
if has_credentials:
    print(f"🌐 Astra DB Endpoint: {astra_db_api_endpoint[:35]}...")
    print(f"🔑 Application Token: {astra_db_application_token[:15]}...")
else:
    print("ℹ️ Astra DB credentials not detected in .env.")
    print("👉 Sign up for free at: https://astra.datastax.com to get your credentials.")
    print("💡 Running in local demonstration mode.")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب فحص وقراءة مفاتيح Astra DB من بيئة العمل بنفسك هنا):



### 2. تجهيز نموذج التضمين والاتصال بمخزن Astra DB
**الفكرة ببساطة:**
بنجهز نموذج التضمين (HuggingFace محلي مجاني، أو OpenAI إن توفر المفتاح)، وبنتصل بكلاس `AstraDBVectorStore` التابع لحزمة `langchain-astradb` مع تحديد اسم المجموعة (`astra_vector_langchain`).

In [ ]:
# initialize embedding model
if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("Using OpenAI Embeddings (text-embedding-3-small)")
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    print("Using local HuggingFace Embeddings (all-MiniLM-L6-v2)")

# connect to AstraDBVectorStore (or fallback in-memory simulator if no credentials)
from langchain_astradb import AstraDBVectorStore

if has_credentials:
    vector_store = AstraDBVectorStore(
        collection_name="astra_vector_langchain",
        embedding=embeddings,
        api_endpoint=astra_db_api_endpoint,
        token=astra_db_application_token,
        namespace="default_keyspace"
    )
    print("✅ Connected to LIVE DataStax Astra DB cluster!")
else:
    from langchain_core.vectorstores import InMemoryVectorStore
    vector_store = InMemoryVectorStore(embeddings)
    print("✅ Initialized vector store demonstration engine.")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تهيئة التضمين والاتصال بـ AstraDBVectorStore بنفسك هنا):



### 3. تجهيز قائمة المستندات المتنوعة مع البيانات الوصفية (Metadata)
**الفكرة ببساطة:**
كما شرح المحاضر في الفيديو، بنجهز مجموعة من 10 مستندات تغطي مواضيع مختلفة (تغريدات عن الإفطار، مشاريع LangChain، تقارير الطقس، وأخبار البورصة) مع تصنيف المصدر (`source`: `tweet`, `news`, `tech_blog`).

In [ ]:
# import core document structure
from langchain_core.documents import Document

# create 10 multi-domain sample documents matching lecture demonstration
sample_documents = [
    Document(
        page_content="I had delicious chocolate chip pancakes and scrambled eggs for breakfast.",
        metadata={"source": "tweet", "topic": "food", "author": "foodie_99"}
    ),
    Document(
        page_content="Building an exciting enterprise generative AI project using LangChain and LangGraph.",
        metadata={"source": "tweet", "topic": "ai", "author": "dev_guru"}
    ),
    Document(
        page_content="The weather forecast for tomorrow indicates cloudy skies with afternoon rain showers.",
        metadata={"source": "news", "topic": "weather", "urgency": "low"}
    ),
    Document(
        page_content="Stock market indices surged today driven by strong quarterly earnings in tech sectors.",
        metadata={"source": "news", "topic": "finance", "urgency": "medium"}
    ),
    Document(
        page_content="A balanced breakfast containing healthy proteins and fruits helps maintain daytime focus.",
        metadata={"source": "health_blog", "topic": "food", "author": "nutritionist"}
    ),
    Document(
        page_content="Stealing from the bank or committing wire fraud is a serious federal crime.",
        metadata={"source": "news", "topic": "legal", "urgency": "high"}
    ),
    Document(
        page_content="DataStax Astra DB provides near-zero latency serverless vector database powered by Apache Cassandra.",
        metadata={"source": "tech_blog", "topic": "ai", "author": "cloud_arch"}
    ),
    Document(
        page_content="Severe tropical storm warnings were issued along coastal regions with high wind speeds.",
        metadata={"source": "news", "topic": "weather", "urgency": "high"}
    ),
    Document(
        page_content="LangChain Expression Language (LCEL) simplifies building streaming conversational RAG pipelines.",
        metadata={"source": "tech_blog", "topic": "ai", "author": "dev_guru"}
    ),
    Document(
        page_content="Central banks announced updated interest rate guidance following the economic policy summit.",
        metadata={"source": "news", "topic": "finance", "urgency": "medium"}
    ),
]

print(f"✅ Prepared {len(sample_documents)} sample documents:")
for idx, doc in enumerate(sample_documents[:4], start=1):
    print(f"  [{idx}] (Source: {doc.metadata['source']:<10} | Topic: {doc.metadata['topic']:<8}): '{doc.page_content[:55]}...'")
print("  ... and 6 more documents.")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب إنشاء قائمة مستندات مع تصنيف الـ metadata بنفسك هنا):



### 4. إضافة المستندات لقاعدة البيانات (`add_documents`)
**الفكرة ببساطة:**
بنرفع المستندات لقاعدة البيانات السحابية عبر `vector_store.add_documents`.
يقوم Astra DB بتحويلها لمتجهات وتخزينها في جدول Cassandra السحابي، وبتلاحظ على لوحة التحكم زيادة الـ Throughput وعدد السجلات إلى 10 فورياً!

In [ ]:
# add documents to the vector store
doc_ids = vector_store.add_documents(documents=sample_documents)

print(f"✅ Ingested {len(doc_ids)} documents into the vector store!")
print("🔑 Sample Generated Record IDs:")
for idx, doc_id in enumerate(doc_ids[:3], start=1):
    print(f"  [{idx}] {doc_id}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب إضافة المستندات واستخراج معرفات السجلات بنفسك هنا):



### 5. البحث الدلالي السحابي (Similarity Search)
**الفكرة ببساطة:**
بنسأل عن حالة الطقس: *"what is the weather forecast?"* ونطلب أفضل 3 مستندات (`k=3`).
المحرك السحابي بيبحث في المتجهات ويرجع أدق النتائج المتعلقة بالطقس.

In [ ]:
# standard similarity search query
weather_query = "What is the weather forecast for tomorrow?"
weather_results = vector_store.similarity_search(weather_query, k=3)

print(f"❓ Query: '{weather_query}'\n")
print(f"🔍 Top {len(weather_results)} Results:\n")
for idx, doc in enumerate(weather_results, start=1):
    print(f"  [{idx}] Source: {doc.metadata.get('source')} | Topic: {doc.metadata.get('topic')}")
    print(f"      Content: '{doc.page_content}'\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تشغيل بحث دلالي بسؤال من اختيارك بنفسك هنا):



### 6. التصفية بالبيانات الوصفية على السحابة (Metadata Filtering)
**الفكرة ببساطة:**
ميزة قواعد البيانات السحابية الحقيقية زي Astra DB هي دعم **Metadata Filtering** فائق السرعة على مستوى محرك قاعدة البيانات.
هنا بنطلب البحث في التغريدات فقط بـ `filter={"source": "tweet"}` واستبعاد أي مقال إخباري حتى لو كان قريباً في المعنى.

In [ ]:
# similarity search with metadata filter
tweet_query = "What projects are people building with LangChain?"

# define filter predicate (dict for AstraDB, callable for fallback engine)
filter_predicate = {"source": "tweet"} if has_credentials else (lambda doc: doc.metadata.get("source") == "tweet")

tweet_results = vector_store.similarity_search(
    query=tweet_query,
    k=2,
    filter=filter_predicate
)

print(f"🔍 Query: '{tweet_query}' (Filter: source == 'tweet')\n")
print(f"✅ Filtered Results count: {len(tweet_results)}\n")
for idx, doc in enumerate(tweet_results, start=1):
    print(f"  [{idx}] Source: {doc.metadata.get('source')} | Author: {doc.metadata.get('author')}")
    print(f"      Content: '{doc.page_content}'\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب البحث مع تطبيق فلتر metadata من اختيارك هنا):



### 7. البحث مع استخراج درجات التطابق (Similarity Search with Score)
**الفكرة ببساطة:**
باستخدام `similarity_search_with_score`، بنحصل على النص المسترجع ومعه نسبة التطابق الدلالي (Score) لتقييم مدى ثقة النتيجة.

In [ ]:
# similarity search returning (document, score) pairs
scored_results = vector_store.similarity_search_with_score("What legal consequences exist for bank theft?", k=2)

print("📊 Results with Relevance Scores:\n")
for idx, (doc, score) in enumerate(scored_results, start=1):
    print(f"  Rank #{idx} | Similarity Score: {score:.4f}")
    print(f"    Topic:   {doc.metadata.get('topic')} | Source: {doc.metadata.get('source')}")
    print(f"    Content: '{doc.page_content}'\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استخراج درجات التطابق وفحص نسبتها بنفسك هنا):



### 8. المسترجعات المتقدمة: عتبة الثقة (Threshold) وتنوع النتائج (MMR)
**الفكرة ببساطة:**
في هذا الدرس استعرض المحاضر نمطين متقدمين لتحويل الفهرس إلى `Retriever`:
1. **`similarity_score_threshold`:** يرفض أي نتيجة ثقتها أقل من نسبة محددة (مثلاً `score_threshold=0.5`).
2. **`mmr` (Maximal Marginal Relevance):** يوازن بين مطابقة السؤال وتنوع الأفكار لمنع التكرار (مقدمة هامة لسكشن 9!).

In [ ]:
# 1. retriever with similarity score threshold (native in AstraDB)
try:
    threshold_retriever = vector_store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 2, "score_threshold": 0.4}
    )
    test_q = "Is stealing from the bank a crime?"
    threshold_docs = threshold_retriever.invoke(test_q)
    print(f"🔍 Threshold Retriever (score >= 0.4) returned {len(threshold_docs)} docs for: '{test_q}'")
    if threshold_docs:
        print(f"   Top Match: '{threshold_docs[0].page_content[:60]}...'\n")
except Exception as e:
    print(f"ℹ️ similarity_score_threshold is supported natively in AstraDB: {e}\n")

# 2. retriever with maximal marginal relevance (MMR)
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "fetch_k": 5}
)

# test mmr retriever
mmr_docs = mmr_retriever.invoke("Tell me about breakfast options")
print(f"🔍 MMR Retriever (Diversity-aware) returned {len(mmr_docs)} docs:")
for idx, d in enumerate(mmr_docs, start=1):
    print(f"   [{idx}] {d.page_content[:65]}...")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تجهيز واستدعاء مسترجع بـ threshold أو MMR بنفسك هنا):



### 9. بناء مسار RAG كامل باستخدام LCEL ونموذج Groq السريع
**الفكرة ببساطة:**
بنربط مسترجع Astra DB السحابي بموديل الذكاء الاصطناعي السريع **Groq** وقالب الـ Prompt الصارم عبر مشغل الربط `|` في LCEL للحصول على إجابة نهائية دقيقة وموثقة.

In [ ]:
# import lcel components and groq chat model
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# helper function to format context chunks
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# context-grounded prompt template
template = """You are a concise enterprise AI assistant.
Answer the user's question using ONLY the provided context. If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# initialize fast Groq LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)

# base retriever
base_retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# construct lcel chain
rag_chain = (
    {"context": base_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# execute test query
rag_query = "What is DataStax Astra DB and what engine powers it?"
answer = rag_chain.invoke(rag_query)

print(f"❓ Question: {rag_query}\n")
print(f"💬 Answer from Astra DB RAG:\n{answer}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب ربط السلسلة وتشغيل استعلام RAG كامل بنفسك هنا):



## 🎯 ملخص المعمل السريع (Lecture 35):
1. **DataStax Astra DB:** قاعدة بيانات متجهات سحابية بدون خوادم (Serverless) مبنية على Apache Cassandra، وتتحمل مليارات المتجهات.
2. **المفتاحان الأساسيان:** للاتصال تحتاج فقط `ASTRA_DB_API_ENDPOINT` و `ASTRA_DB_APPLICATION_TOKEN`.
3. **التصفية السحابية السريعة:** تدعم `filter={"key": "val"}` مباشرة على مستوى محرك قاعدة البيانات.
4. **المسترجعات المتقدمة:** تدعم البحث بالعتبة (`similarity_score_threshold`) والبحث المتنوع بـ `mmr`.
5. **المحاضرة القادمة (Lecture 36):** العمل مع وحش المتجهات السحابي الآخر: **Pinecone Vector Database**!